# DS-1000

This example shows how to evaluate a `genlm.control` model on the DS-1000 domain.

* **Task**: Generate valid and executable Python code.
* **Data**: DS-1000 dataset (Lai et al., 2022).

## Setup

First, install the dependencies for this domain. In the root directory, run:    

```bash
pip install -e .[ds1000]
```

## Usage 

### Initialize the dataset and evaluator

In [2]:
from genlm.eval.domains.ds1000 import (
    DS1000Dataset, DS1000Evaluator
)

In [6]:
dataset = DS1000Dataset.from_hf(
    split="test",
    libraries=None,
    perturbation_types=None,
    max_instances=8,
    shuffle=False,
)

print("Instances loaded:", len(dataset))
evaluator = DS1000Evaluator()

Instances loaded: 8


### Inspect dataset

In [7]:
first = next(iter(dataset))
print("Instance ID:", first.instance_id)
print("Library:", first.metadata.get("library"))
print("Perturbation:", first.metadata.get("perturbation_type"))
print("Prompt preview:\n", (first.prompt[:500] + "...") if len(first.prompt) > 500 else first.prompt)

Instance ID: 0
Library: Pandas
Perturbation: Origin
Prompt preview:
 Problem:
I have the following DataFrame:
    Col1  Col2  Col3  Type
0      1     2     3     1
1      4     5     6     1
2      7     8     9     2
3    10    11    12     2
4    13    14    15     3
5    16    17    18     3


The DataFrame is read from a CSV file. All rows which have Type 1 are on top, followed by the rows with Type 2, followed by the rows with Type 3, etc.
I would like to shuffle the order of the DataFrame's rows according to a list. \
For example, give a list [2, 4, 0, 3, 1...


## Model Adaptor

In [ ]:
from genlm.control import PromptedLLM, AWRS
from genlm.eval import ModelOutput, ModelResponse
from genlm.eval.domains.ds1000 import (
    DS1000RuntimeNoErrorPotential, default_prompt_formatter
)

# Load an LLM
LLM = PromptedLLM.from_name("gpt2")


async def model(instance, output_dir, replicate):
    # Set the prompt for the LLM.
    LLM.prompt_ids = default_prompt_formatter(
        LLM.model.tokenizer, instance, use_chat_format=False
    )

    # Define a potential that ensures the code throws no error
    potential = DS1000RuntimeNoErrorPotential(
        code_context=instance.code_context,
        timeout_seconds=10,
    ).coerce(LLM)

    # Define an adaptive weighted rejection sampler to sample tokens from the constrained model.
    sampler = AWRS(LLM, potential)

    # Run SMC to sample sequences from the constrained model.
    sequences = await sampler.smc(
        n_particles=5,
        ess_threshold=0.5,
        max_tokens=100,
    )

    # Return the sampled sequences and their probabilities as a ModelOutput.
    return ModelOutput(
        responses=[
            ModelResponse(response=sequence, weight=prob)
            for sequence, prob in sequences.decoded_posterior.items()
        ],
    )

### Run the evaluation

In [ ]:
from genlm.eval import run_evaluation

results = await run_evaluation(
    dataset=dataset,
    model=model,
    evaluator=evaluator,
    max_instances=2,
    n_replicates=1,
    verbosity=1,
    # output_dir="ds1000_results", optionally save the results to a directory
)

/cluster/home/skiegeland/genlm-control/genlm/control/potential/product.py:80: RuntimeWarning: Common vocabulary (256 tokens) is less than 10.0% of p1's (PromptedLLM(prompt=[b'Problem', b':', b'\n', b'I', b' have', b' the', b' following', b' Data', b'Frame', b':', b'\n', b' ', b' ', b' ', b' Col', b'1', b' ', b' Col', b'2', b' ', b' Col', b'3', b' ', b' Type', b'\n', b'0', b' ', b' ', b' ', b' ', b' ', b' 1', b' ', b' ', b' ', b' ', b' 2', b' ', b' ', b' ', b' ', b' 3', b' ', b' ', b' ', b' ', b' 1', b'\n', b'1', b' ', b' ', b' ', b' ', b' ', b' 4', b' ', b' ', b' ', b' ', b' 5', b' ', b' ', b' ', b' ', b' 6', b' ', b' ', b' ', b' ', b' 1', b'\n', b'2', b' ', b' ', b' ', b' ', b' ', b' 7', b' ', b' ', b' ', b' ', b' 8', b' ', b' ', b' ', b' ', b' 9', b' ', b' ', b' ', b' ', b' 2', b'\n', b'3', b' ', b' ', b' ', b' 10', b' ', b' ', b' ', b' 11', b' ', b' ', b' ', b' 12', b' ', b' ', b' ', b' ', b' 2', b'\n', b'4', b' ', b' ', b' ', b' 13', b' ', b' ', b' ', b' 14', b' ', b' ', b' ', b' 1

Yuhang Lai, Chengxi Li, Yiming Wang, Tianyi Zhang, Ruiqi Zhong, Luke Zettlemoyer, Wen-Tau Yih, Daniel Fried, Sida Wang, and Tao Yu. DS-1000: A natural and reliable benchmark for data science code generation. arXiv preprint arXiv:2211.11501, 2022. URL https://arxiv.org/abs/2211.11501